In [1]:
db = "./harvard/harvard.db"

In [2]:
%load_ext sql
%config SqlMagic.feedback=False
%sql sqlite:///{db}?check_same_thread=false
%sql SELECT name FROM sqlite_master WHERE type='table';

 * sqlite:///./harvard/harvard.db?check_same_thread=false


name
students
enrollments
courses
satisfies
requirements


In [ ]:
%%sql
CREATE INDEX "enrollments_student_course_idx" ON "enrollments" ("student_id", "course_id");
CREATE INDEX "courses_semester_dept_num_idx" ON "courses" ("semester", "department", "number");
CREATE INDEX "satisfies_course_requirement_idx" ON "satisfies" ("course_id", "requirement_id");

In [3]:
%sql SELECT * FROM "sqlite_master";

 * sqlite:///./harvard/harvard.db?check_same_thread=false


type,name,tbl_name,rootpage,sql
table,students,students,2,"CREATE TABLE ""students"" ( ""id"" INTEGER, ""name"" TEXT NOT NULL, PRIMARY KEY(""id""))"
table,enrollments,enrollments,3,"CREATE TABLE ""enrollments"" ( ""id"" INTEGER, ""student_id"" INTEGER, ""course_id"" INTEGER, PRIMARY KEY(""id""), FOREIGN KEY(""student_id"") REFERENCES ""students""(""id""), FOREIGN KEY(""course_id"") REFERENCES ""courses""(""id""))"
table,courses,courses,4,"CREATE TABLE ""courses"" ( ""id"" INTEGER, ""department"" TEXT NOT NULL, ""number"" INTEGER NOT NULL, ""semester"" TEXT NOT NULL, ""title"" TEXT NOT NULL, PRIMARY KEY(""id""))"
table,satisfies,satisfies,5,"CREATE TABLE ""satisfies"" ( ""id"" INTEGER, ""course_id"" INTEGER, ""requirement_id"" INTEGER, PRIMARY KEY(""id""), FOREIGN KEY(""course_id"") REFERENCES ""courses""(""id""), FOREIGN KEY(""requirement_id"") REFERENCES ""requirements""(""id""))"
table,requirements,requirements,6,"CREATE TABLE ""requirements"" ( ""id"" INTEGER, ""name"" TEXT NOT NULL, PRIMARY KEY(""id""))"
index,enrollments_student_course_idx,enrollments,6304,"CREATE INDEX ""enrollments_student_course_idx"" ON ""enrollments"" (""student_id"", ""course_id"")"
index,courses_semester_dept_num_idx,courses,11941,"CREATE INDEX ""courses_semester_dept_num_idx"" ON ""courses"" (""semester"", ""department"", ""number"")"
index,satisfies_course_requirement_idx,satisfies,12161,"CREATE INDEX ""satisfies_course_requirement_idx"" ON ""satisfies"" (""course_id"", ""requirement_id"")"


In [4]:
%%sql
EXPLAIN QUERY PLAN
SELECT "courses"."title", "courses"."semester"
FROM "enrollments"
JOIN "courses" ON "enrollments"."course_id" = "courses"."id"
JOIN "students" ON "enrollments"."student_id" = "students"."id"
WHERE "students"."id" = 3;

 * sqlite:///./harvard/harvard.db?check_same_thread=false


id,parent,notused,detail
4,0,45,SEARCH students USING INTEGER PRIMARY KEY (rowid=?)
7,0,56,SEARCH enrollments USING COVERING INDEX enrollments_student_course_idx (student_id=?)
12,0,45,SEARCH courses USING INTEGER PRIMARY KEY (rowid=?)


In [5]:
%%sql
EXPLAIN QUERY PLAN
SELECT "id", "name"
FROM "students"
WHERE "id" IN (
    SELECT "student_id"
    FROM "enrollments"
    WHERE "course_id" = (
        SELECT "id"
        FROM "courses"
        WHERE "courses"."department" = 'Computer Science'
        AND "courses"."number" = 50
        AND "courses"."semester" = 'Fall 2023'
    )
);

 * sqlite:///./harvard/harvard.db?check_same_thread=false


id,parent,notused,detail
2,0,91,SEARCH students USING INTEGER PRIMARY KEY (rowid=?)
6,0,0,LIST SUBQUERY 2
9,6,216,SCAN enrollments
14,6,0,SCALAR SUBQUERY 1
18,14,54,SEARCH courses USING COVERING INDEX courses_semester_dept_num_idx (semester=? AND department=? AND number=?)
33,6,0,CREATE BLOOM FILTER


In [6]:
%%sql
EXPLAIN QUERY PLAN
SELECT "courses"."id", "courses"."department", "courses"."number", "courses"."title", COUNT(*) AS "enrollment"
FROM "courses"
JOIN "enrollments" ON "enrollments"."course_id" = "courses"."id"
WHERE "courses"."semester" = 'Fall 2023'
GROUP BY "courses"."id"
ORDER BY "enrollment" DESC;

 * sqlite:///./harvard/harvard.db?check_same_thread=false


id,parent,notused,detail
8,0,216,SCAN enrollments
10,0,45,SEARCH courses USING INTEGER PRIMARY KEY (rowid=?)
15,0,0,USE TEMP B-TREE FOR GROUP BY
56,0,0,USE TEMP B-TREE FOR ORDER BY


In [7]:
%%sql
EXPLAIN QUERY PLAN
SELECT "courses"."id", "courses"."department", "courses"."number", "courses"."title"
FROM "courses"
WHERE "courses"."department" = 'Computer Science'
AND "courses"."semester" = 'Spring 2024';

 * sqlite:///./harvard/harvard.db?check_same_thread=false


id,parent,notused,detail
3,0,62,SEARCH courses USING INDEX courses_semester_dept_num_idx (semester=? AND department=?)


In [8]:
%%sql
EXPLAIN QUERY PLAN
SELECT "requirements"."name"
FROM "requirements"
WHERE "requirements"."id" = (
    SELECT "requirement_id"
    FROM "satisfies"
    WHERE "course_id" = (
        SELECT "id"
        FROM "courses"
        WHERE "title" = 'Advanced Databases'
        AND "semester" = 'Fall 2023'
    )
);

 * sqlite:///./harvard/harvard.db?check_same_thread=false


id,parent,notused,detail
2,0,33,SEARCH requirements USING INTEGER PRIMARY KEY (rowid=?)
5,0,0,SCALAR SUBQUERY 2
9,5,56,SEARCH satisfies USING COVERING INDEX satisfies_course_requirement_idx (course_id=?)
12,5,0,SCALAR SUBQUERY 1
17,12,63,SEARCH courses USING INDEX courses_semester_dept_num_idx (semester=?)


In [9]:
%%sql
EXPLAIN QUERY PLAN
SELECT "requirements"."name", COUNT(*) AS "courses"
FROM "requirements"
JOIN "satisfies" ON "requirements"."id" = "satisfies"."requirement_id"
WHERE "satisfies"."course_id" IN (
    SELECT "course_id"
    FROM "enrollments"
    WHERE "enrollments"."student_id" = 8
)
GROUP BY "requirements"."name";

 * sqlite:///./harvard/harvard.db?check_same_thread=false


id,parent,notused,detail
7,0,102,SEARCH satisfies USING COVERING INDEX satisfies_course_requirement_idx (course_id=?)
11,0,0,LIST SUBQUERY 1
14,11,56,SEARCH enrollments USING COVERING INDEX enrollments_student_course_idx (student_id=?)
22,11,0,CREATE BLOOM FILTER
31,0,45,SEARCH requirements USING INTEGER PRIMARY KEY (rowid=?)
34,0,0,USE TEMP B-TREE FOR GROUP BY


In [10]:
%%sql
EXPLAIN QUERY PLAN
SELECT "department", "number", "title"
FROM "courses"
WHERE "title" LIKE 'History%'
AND "semester" = 'Fall 2023';

 * sqlite:///./harvard/harvard.db?check_same_thread=false


id,parent,notused,detail
3,0,63,SEARCH courses USING INDEX courses_semester_dept_num_idx (semester=?)
